# Resultados y figuras para la presentación
## YOLO + SAHI sobre VisDrone — FLISOL 2026

Produce **todo el material** de la charla:

| Figura | Qué muestra | Requiere modelo VisDrone |
|---|---|---|
| `00_pipeline_sahi.png` | Diagrama del pipeline | no |
| `06_size_distribution.png` | Tamaños de objeto (mayoría small) | no |
| `07_objects_per_image.png` | Densidad de objetos | no |
| `08_class_distribution.png` | Frecuencia de clases | no |
| `09_slice_grid.png` | Distribución de slices con/sin overlap | no |
| `03_slice_tradeoff.png` | Detecciones vs tiempo | no |
| `05_per_class.png` | Detecciones por clase | no |
| `01_map_comparison.png` | mAP global YOLO vs SAHI | **sí** |
| `02_ap_por_tamano.png` | AP por tamaño ⭐ | **sí** |
| `04_postprocess.png` | NMS vs NMM vs GREEDYNMM | **sí** |
| `10_per_class_ap.png` | AP por clase | **sí** |
| `gallery/*.jpg`, `segmentation/*.jpg` | Comparativas visuales | no |

> ⚠️ El mAP real necesita `00_entrenamiento.ipynb`. Sin él, las figuras 'no' se generan igual.
> ⚙️ Activa la GPU (T4).

---
## 1. Setup

In [ ]:
!pip install -q ultralytics sahi pycocotools matplotlib
!git clone https://github.com/jossuema/yolo-sahi-flisol2026.git || echo 'ya clonado'
%cd yolo-sahi-flisol2026
import os, sys, glob; sys.path.insert(0, 'src')
from IPython.display import Image, display
import torch; print('GPU:', torch.cuda.is_available())

---
## 2. Datos + modelo

Descarga VisDrone, lo convierte a COCO GT y recupera el modelo entrenado desde Drive si existe.

In [ ]:
import shutil
try:
    from google.colab import drive; drive.mount('/content/drive')
    src = '/content/drive/MyDrive/flisol2026/visdrone_yolo11s.pt'
    if os.path.exists(src):
        os.makedirs('weights', exist_ok=True); shutil.copy(src, 'weights/visdrone_yolo11s.pt')
        print('Peso VisDrone recuperado de Drive')
except Exception as e:
    print('(sin Drive)', e)

!python scripts/download_visdrone.py
MODEL = 'weights/visdrone_yolo11s.pt' if os.path.exists('weights/visdrone_yolo11s.pt') else 'yolo11s.pt'
HAS_VISDRONE = os.path.exists('weights/visdrone_yolo11s.pt')
print('Modelo:', MODEL, '| mAP real disponible:', HAS_VISDRONE)
if not HAS_VISDRONE:
    print('AVISO: sin modelo VisDrone el mAP saldra ~0 (zero-shot). Entrena con 00_entrenamiento.ipynb.')

---
## 3. Estadísticas del dataset (explicativas, sin modelo)

Por qué VisDrone es el caso ideal de SAHI: tamaños de objeto, densidad y clases.

In [ ]:
!python src/stats_dataset.py
for f in ['06_size_distribution', '07_objects_per_image', '08_class_distribution']:
    display(Image(f'outputs/figures/{f}.png'))

---
## 4. Distribución de slices con/sin overlap (sin modelo)

Cómo SAHI reparte la imagen en mosaicos. Las zonas oscuras = solapamiento entre slices.

In [ ]:
!python src/viz_slices.py
display(Image('outputs/figures/09_slice_grid.png'))

---
## 5. Benchmark mAP — YOLO vs YOLO + SAHI (+ AP por clase)

Evalúa con `pycocotools`. Genera `benchmark_map.csv` y `per_class_ap.csv`.
Si ves la advertencia de desalineación, el modelo NO está entrenado en VisDrone.

In [ ]:
!python src/evaluate.py --model $MODEL --limit 100
import pandas as pd
pd.read_csv('outputs/benchmark_map.csv') if os.path.exists('outputs/benchmark_map.csv') else 'sin resultados'

---
## 6. Trade-off del tamaño de slice

In [ ]:
!python src/slice_sweep.py --model $MODEL --limit 15 --sizes 320 512 640 768 1024

---
## 7. Comparación de postprocesamiento (NMS / NMM / GREEDYNMM)

In [ ]:
!python src/compare_postprocess.py --model $MODEL --limit 60

---
## 8. Análisis por clase (conteo de detecciones)

Salto concreto en `pedestrian`, `people`, `motor` — el caso de seguridad ciudadana.

In [ ]:
!python src/analyze_classes.py --model $MODEL --limit 60

---
## 9. Generar TODAS las figuras y previsualizarlas

In [ ]:
import importlib, figures; importlib.reload(figures)
figures.all_figures()
for f in sorted(glob.glob('outputs/figures/*.png')):
    print(f); display(Image(f))

---
## 10. Galería de detección antes/después

In [ ]:
!python src/gallery.py --model $MODEL --limit 30 --top 8
for f in sorted(glob.glob('outputs/gallery/*.jpg'))[:4]:
    print(f); display(Image(f))

---
## 11. Segmentación antes/después (varias imágenes)

Paneles de segmentación de instancias YOLO-seg vs YOLO-seg + SAHI, ordenados por ganancia.

In [ ]:
!python src/segment.py --limit 25 --top 6
for f in sorted(glob.glob('outputs/segmentation/seg_*.jpg'))[:4]:
    print(f); display(Image(f))

---
## 12. Empaquetar todo para las slides

In [ ]:
!zip -r resultados_slides.zip outputs/figures outputs/gallery outputs/segmentation outputs/*.csv
try:
    from google.colab import files; files.download('resultados_slides.zip')
except Exception as e:
    print('Descarga manual: resultados_slides.zip', e)